# Surveillance des Plans d'Eau et Ressources Hydrologiques

## Introduction
La gestion de l'eau est cruciale en RDC, tant pour le transport que pour la biodiversité. Ce notebook utilise l'indice MNDWI pour détecter précisément les lacs, rivières et zones humides, tout en minimisant les erreurs causées par le relief ou le bâti urbain.

## Objectifs
*   **Inventaire de l'eau** : Isoler les corps de surface.
*   **Limitation des faux positifs** : Utiliser le SWIR pour annuler l'impact des ombres.
*   **Suivi des berges** : Déterminer la géométrie exacte des plans d'eau.

## Méthodologie
1.  **Setup** : Installation des outils géospatiaux.
2.  **Acquisition** : Images Sentinel-2 (Bandes Vert et SWIR1).
3.  **Analyse MNDWI** : Application de l'Indice d'Eau Normalisé Modifié.
4.  **Visualisation** : Carte des ressources hydriques.

In [ ]:
# ====================================================
# ÉTAPE 1 : Configuration
# ====================================================
!pip install geemap earthengine-api rasterio matplotlib -q

import ee, geemap, rasterio
import numpy as np
import matplotlib.pyplot as plt

try: ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

print("✅ Système prêt")

## Zone d'Étude (ROI)
Alignement sur la zone d'étude régionale.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d\'étude')
Map

## Acquisition des Données
Le MNDWI utilise la bande Verte (B3) et la bande SWIR1 (B11).

In [ ]:
# ====================================================
# ÉTAPE 3 : Données Sentinel-2
# ====================================================
image = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(roi).median().clip(roi))

geemap.ee_export_image(image.select(['B3', 'B11']), 'water.tif', scale=30, region=roi)

## Calcul MNDWI
L'indice MNDWI est particulièrement efficace pour extraire l'eau tout en supprimant le signal provenant des bâtiments ou des grandes ombres portées par le relief montagneux.

In [ ]:
# ====================================================
# ÉTAPE 4 : Algorithme MNDWI
# ====================================================
with rasterio.open('water.tif') as src: data = src.read().astype(np.float32)
green, swir = data[0], data[1]
mndwi = (green - swir) / (green + swir + 1e-8)

water_mask = mndwi > 0

plt.figure(figsize=(10, 8))
plt.imshow(water_mask, cmap='Blues_r')
plt.title("Détection Précise des Plans d'Eau (MNDWI)")
plt.axis('off')
plt.show()